In [1]:
%load_ext autoreload
%autoreload 3 --print --log

# 本地 HM3D：RGB-D 与实例语义融合

使用 `/R/Datasets/3DGS/hm3dv2` 的本地导出格式，读取 3 个真实帧，不启动 Habitat。
可通过 `GRAPHMAP_HM3D_ROOT` / `GRAPHMAP_HM3D_SCENE` 修改路径。

已核对 FNAV 导出程序与场景 0000 的配置哈希：640×640、HFOV 64°、深度 uint16 毫米；RGB/Depth/Semantic 对齐。
`runtime_log.pkl` 中是 **floor-relative graphmap 相机位姿**，因此不再次叠加相机外参、减楼层高度或换 Habitat 轴。
语义值是导出的实例 ID；本例不把它们当作类别名称。

不同来源/版本不能只换路径：需确认下方标定、深度类型和位姿元数据。本例只读取用户提供的可信 pickle。

In [2]:
import os
import json
import pickle
from pathlib import Path
import numpy as np
from scipykit.task.cv import *
from graphmap.pose import Pose
from graphmap.sensor import CameraIntrinsic
from graphmap.adapters import rgbd_observation
from graphmap.adapters.hm3d import load_hm3d_frame
from graphmap.environment import Environment

root = Path(os.environ.get("GRAPHMAP_HM3D_ROOT", "/R/Datasets/3DGS/hm3dv2"))
scene = root / os.environ.get("GRAPHMAP_HM3D_SCENE", "0000")
metadata = json.loads((scene / "navigation_replay.json").read_text())
signature = metadata["signature"]
assert signature["coordinate_frames"]["export_pose"] == "floor_relative_rgb_camera"
# 固定到已验证的标定，未知配置应由使用者明确检查，而不是静默猜测。
expected_config_hash = "7570bfa76cd07c69eee812f2b170ac0ef492a190c617e0eb897d797be7617bec"
if signature["config_sha256"]["simulator_config"] != expected_config_hash:
    raise ValueError("此导出的配置未验证，请核对相机参数后修改此单元格")
with (scene / "runtime_log.pkl").open("rb") as stream:
    camera_log = pickle.load(stream)
frame_ids = sorted(int(path.name.split("_")[0]) for path in scene.glob("*_rgb.png"))[:3]
assert len(frame_ids) == 3
camera = CameraIntrinsic.from_sensor(width=640, height=640, hfov=np.deg2rad(64), vfov=np.deg2rad(64))
print("scene:", scene.name, "frames:", frame_ids, "focal:", camera.fx)

scene: 0000 frames: [0, 2, 3] focal: 512.1070492931361


In [3]:
first = load_hm3d_frame(scene, frame_ids[0])
fig, axes = subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(first["rgb"])
axes[0].set_title("RGB")
axes[1].imshow(first["depth"] * 0.001, cmap="viridis")
axes[1].set_title("Axial depth (m)")
if first["semantic"] is not None:
    axes[2].imshow(first["semantic"], cmap="tab20")
    axes[2].set_title("Semantic instance ID")
else:
    axes[2].text(0.5, 0.5, "No semantic labels", ha="center")
for ax in axes:
    ax.set(xlabel="w", ylabel="h")
disp(fig, '1')
plt.close(fig)

/gray/Storage/Repositories/graphmap/examples/assets/03_hm3d_semantics.ipynb/1.png

## 使用导出的相机 pose 融合三帧
`stride=16` 控制演示数据量。timestamp 使用示例会话相对秒，仅表达融合顺序，不代表原始采集时间。
没有独立 body pose 时，适配器用相机 pose 作为 Environment 的当前参考 pose；这不是机器人底盘位置。

In [4]:
env = Environment(resolution=0.05, enable_2d_map=True, side_length=40)
for order, frame_id in enumerate(frame_ids):
    frame = load_hm3d_frame(scene, frame_id)
    record = camera_log[frame_id]
    T_world_camera = Pose(position=record["curr_pos"], rotation=record["curr_rot"],
                          wrd_frame="world", ego_frame="camera")
    attributes = {} if frame["semantic"] is None else {"semantic_id": frame["semantic"]}
    observation = rgbd_observation(
        frame["depth"], intrinsic=camera, pose_camera=T_world_camera,
        rgb=frame["rgb"], attributes=attributes, depth_scale=0.001,
        depth_kind="axial", depth_range=(0.05,40), stride=16, timestamp=order * 0.1,
    )
    env.observe(**observation)
    print(frame_id, "sampled:", len(observation["points_world"]), "map points:", env.N)
assert env.N > 0
print(env.status())

0 sampled: 1600 map points: 1114
2 sampled: 1590 map points: 1988
3 sampled: 1591 map points: 2824
{'num_points': 2824, 'num_db_tables': 1, 'num_db_records': 27, 'num_db_links': 2824, 'db_tables': {'semantic_id': {'records': 27, 'links': 2824}}, 'frame_count': 3, 'has_2d_map': True, 'last_pose': 'Pose(pos=[-0.986 -7.641  1.014], euler_deg=[  0.     10.    156.774], scale=1) [world <- camera]'}


In [5]:
semantic = env.get_index_values("semantic_id", default=-1)
counts = {int(value): semantic.count(value) for value in set(semantic)}
print("instance id -> point count:", dict(sorted(counts.items())[:12]))
if "semantic_id" in env.db:
    print("unique values:", env.index_db("semantic_id").size,
          "point links:", env.index_db("semantic_id").num_links)
# 最后一帧采样后的点/颜色/语义仍一一对齐。
h, w = observation["pixel_indices"].T
np.testing.assert_array_equal(observation["colors"], frame["rgb"][h,w])
if frame["semantic"] is not None:
    assert observation["extra_keys"]["semantic_id"].get_batch(range(len(h))) == frame["semantic"][h,w].tolist()

instance id -> point count: {0: 55, 112: 21, 116: 21, 128: 4, 129: 2, 130: 8, 132: 8, 133: 3, 147: 9, 148: 158, 149: 3, 155: 69}
unique values: 27 point links: 2824


In [6]:
fig = plt.figure(figsize=(11, 4), dpi=120)
ax3d = fig.add_subplot(121, projection="3d")
ax3d.scatter(*env.infopoints.xyz.T, c=env.infopoints.color[:,:3] / 255, s=2)
ax3d.set(xlabel="world x (m)", ylabel="world y (m)", zlabel="world z (m)")
axmap = fig.add_subplot(122)
axmap.imshow(env.map2d.grid[:,:,0], origin="lower", vmin=-1, vmax=1)
axmap.set(title="All observed points projected to XY", xlabel="w / world y index", ylabel="h / world x index")
# 只缩放显示窗口，不转置数组；h 对应 x，w 对应 y。
occupied_hw = env.map2d.world_to_grid(env.infopoints.xyz[:, :2])
h_min, w_min = occupied_hw.min(axis=0) - 10
h_max, w_max = occupied_hw.max(axis=0) + 10
axmap.set_xlim(w_min, w_max)
axmap.set_ylim(h_min, h_max)
fig.tight_layout()
disp(fig, '2')
plt.close(fig)

/gray/Storage/Repositories/graphmap/examples/assets/03_hm3d_semantics.ipynb/2.png

当前地图是三帧观测的投影占用示意，包含地面点；导航前应先按高度、地面判定等规则区分障碍与地面。此库的 `Environment.observe` 当前不自动清除动态物体。

扩展到其他数据源时：只替换读取层，明确单位/坐标/pose 方向后，复用 `rgbd_observation` 和 `Environment`。逐像素特征图也可作为新的 `attributes` 表输入。